# Toy Feature Math

This is the same kind of simple feature math the orchestrator does before calling inference. It intentionally uses only the Python standard library.

In [1]:
from statistics import mean, pstdev

closes = [100.0, 100.5, 101.0, 100.8, 101.2, 101.7]
volumes = [1000, 1200, 900, 1100, 1300, 1600]

latest_close = closes[-1]
latest_volume = volumes[-1]

def pct_return(values, period):
    if len(values) <= period:
        return 0.0
    return (values[-1] / values[-period - 1]) - 1.0

features = {
    "close": latest_close,
    "volume": latest_volume,
    "returns_1": pct_return(closes, 1),
    "returns_5": pct_return(closes, 5),
    "moving_average_distance": (latest_close / mean(closes)) - 1.0,
    "volatility": pstdev(closes) / mean(closes),
    "volume_zscore": (latest_volume - mean(volumes)) / pstdev(volumes),
}

features

{'close': 101.7,
 'volume': 1600,
 'returns_1': 0.004940711462450675,
 'returns_5': 0.017000000000000126,
 'moving_average_distance': 0.008261731658955718,
 'volatility': 0.005297825360833255,
 'volume_zscore': 1.8380365552345197}

In [2]:
def toy_predict(features):
    score = (features["returns_5"] * 3.0) + (features["moving_average_distance"] * 2.0) - min(features["volatility"], 0.05)
    probability_up = max(0.05, min(0.95, 0.5 + score))
    if probability_up > 0.58:
        action = "buy"
    elif probability_up < 0.42:
        action = "sell"
    else:
        action = "hold"
    return {"action": action, "probability_up": probability_up, "score": score}

toy_predict(features)

{'action': 'hold',
 'probability_up': 0.5622256379570786,
 'score': 0.062225637957078556}